In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# El nombre del experimento debe terminar en "control-2"
usuario = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{usuario}/control-2")

features_df = spark.table("wine_quality_features_raw").toPandas()
labels_df = spark.table("wine_quality_labels").toPandas()
data = features_df.merge(labels_df, on="wine_id").drop(columns=["wine_id"])

X = data.drop(columns=["target"])
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelos = [
    ("modelo-run-1", LogisticRegression(max_iter=1000)),
    ("modelo-run-2", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ("modelo-run-3", RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)),
    ("modelo-run-4", GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, random_state=42)),
    ("modelo-run-5", SVC(kernel="rbf", C=1.0, probability=True, random_state=42)),
]

for run_name, modelo in modelos:
    with mlflow.start_run(run_name=run_name):
        modelo.fit(X_train, y_train)
        preds = modelo.predict(X_test)
        proba = modelo.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        auc = roc_auc_score(y_test, proba)

        mlflow.log_param("modelo", type(modelo).__name__)
        mlflow.log_params(modelo.get_params())
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("auc", auc)

        mlflow.sklearn.log_model(modelo, "model")

        print(f"{run_name} ({type(modelo).__name__}): acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}")